# Jack The Learner - Colab Training

**A humanoid robot brain with 105M parameters - ONE unified architecture.**

## Setup
1. **Runtime → Change runtime type → T4 GPU** (or A100 for faster)
2. Run cells in order
3. Checkpoints save to Google Drive automatically

## Training Pipeline
```
Phase 0 (15-30 min)   Phase 1 (4-12 hrs)    Phase 2 (2-4 hrs)
Learn Physics    →    Learn Walking     →    Learn from Demos
SymPy ground truth    RL with safeguards    Imitation learning
+ EWC Fisher          + Replay buffer       + All safeguards
```

## Architecture: UnifiedBrain (105M params)
- Full TD-MPC2 WorldModel
- Full HAC HierarchicalPlanner (20 skills)
- LLaMA-style Transformer backbone
- Cross-modal fusion (vision, proprio, touch, language)

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/JackTheLearner/checkpoints', exist_ok=True)
print('Google Drive mounted!')

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q sympy tqdm
print('Dependencies installed!')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone repository and link checkpoints to Drive
%cd /content
!rm -rf JackTheLearner 2>/dev/null
!git clone https://github.com/JannoLouwrens/JackTheLearner.git
%cd JackTheLearner

# Symlink checkpoints to Drive - saves go directly to Drive!
!rm -rf checkpoints 2>/dev/null
!ln -s /content/drive/MyDrive/JackTheLearner/checkpoints checkpoints

print('\nCheckpoints linked to Google Drive!')
print('All saves go directly to Drive - safe from disconnects!')

---
## 2. Phase 0: Learn Physics (15-30 min on GPU)

UnifiedBrain learns F=ma, torque, energy, momentum from SymPy ground truth.

**Safeguards prepared:**
- EWC Fisher information computed (protects physics knowledge)
- Replay buffer saved (mix into later phases)

In [ ]:
# Quick test (2-3 min)
!python RobustTrainer.py --phase 0 --epochs 5

In [ ]:
# Full Phase 0 (15-30 min on T4 GPU)
!python RobustTrainer.py --phase 0 --epochs 50
print('\nPhase 0 complete! Physics knowledge learned.')

---
## 3. Phase 1: Learn Walking (4-12 hours)

RL training with **safeguards active**:
- 20% of each batch = Phase 0 physics data (replay buffer)
- EWC penalty protects physics weights
- Multi-rate learning (backbone slower, heads faster)

In [ ]:
# Check Phase 0 checkpoint exists
!ls -la checkpoints/*.pt 2>/dev/null || echo 'No checkpoints yet - run Phase 0 first!'

In [ ]:
# Phase 1: RL Walking (requires gymnasium + mujoco)
!pip install -q gymnasium mujoco
!python RobustTrainer.py --phase 1 --epochs 500
print('\nPhase 1 complete! Robot can walk.')

---
## 4. Phase 2: Learn from Demos (2-4 hours)

Imitation learning with flow matching. All safeguards remain active.

In [ ]:
# Phase 2: Imitation Learning
!python RobustTrainer.py --phase 2 --epochs 100
print('\nPhase 2 complete! Natural movement learned.')

---
## 5. Verify Physics Preservation

In [ ]:
# Test that physics knowledge survived all phases
!python RobustTrainer.py --phase 0 --epochs 0 --verify

In [ ]:
# List all checkpoints (on Drive)
!ls -lh checkpoints/

---
## Checkpoint Reference

| Phase | Checkpoint | Description |
|-------|------------|-------------|
| 0 | `phase0_best.pt` | Physics knowledge |
| 0 | `ewc_state.pt` | Fisher information (weight importance) |
| 0 | `replay_buffer.pt` | Physics samples for replay |
| 1 | `phase1_best.pt` | Best RL checkpoint |
| 1 | `phase1_latest.pt` | Latest (for resume) |
| 2 | `phase2_best.pt` | Final trained brain |

## Resume After Disconnect
1. Re-run setup cells (they reconnect Drive symlink)
2. Training auto-resumes from latest checkpoint
3. Your progress is safe on Google Drive!

---
**Model:** UnifiedBrain - 105M parameters (~420MB)  
**Author:** Janno Louwrens